# Agent Instrumentation with OTel

**Instrumentation** is the practice of emitting telemetry — traces, metrics, and logs — from a running application so you can see what it actually did, not just what you think it did.

**Why instrument an agent?** Agent behavior is non-deterministic and spans many steps: model calls, tool invocations, retries, and subagent hand-offs. Instrumentation is often the only practical way to:

- Understand *why* an agent produced a given result
- Debug failures and unexpected loops
- Track latency, token usage, and cost
- Build evaluation and monitoring on top of real execution data

**Why OpenTelemetry (OTel)?** OTel is the open, vendor-neutral standard for telemetry. Because it's a standard rather than a proprietary SDK, the same instrumentation can be exported to any compatible backend (LangSmith, Phoenix, Jaeger, Grafana, etc.) without rewriting your code.

## Install Required Packages
```sh
uv pip install \
  opentelemetry-api \
  opentelemetry-sdk \
  opentelemetry-exporter-otlp \
  openinference-instrumentation-langchain
```

In [ ]:
from collections.abc import Sequence
from pathlib import Path
from typing import Literal

import polars as pl
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from loguru import logger
from openinference.instrumentation.langchain import LangChainInstrumentor
from opentelemetry import trace
from opentelemetry.sdk.trace import ReadableSpan, TracerProvider
from opentelemetry.sdk.trace.export import (
    SimpleSpanProcessor,
    SpanExporter,
    SpanExportResult,
)

from chain_reaction.config import ModelName, get_chat_model
from chain_reaction.utils import get_last_message

# Set up tracing (export to a file)

In [ ]:
class JSONLFileExporter(SpanExporter):
    """Export OpenTelemetry spans to a file as newline-delimited JSON.

    Each exported span is serialized to a single JSON object and appended on its
    own line (JSONL format), allowing traces to be streamed to disk and parsed
    incrementally one span at a time.

    Attributes:
        _path (Path): Filesystem path of the JSONL file spans are appended to.
    """

    def __init__(self, path: str) -> None:
        """Initialize the exporter.

        Args:
            path: Filesystem path to append exported spans to. The file is
                created on first write and opened in append mode thereafter.
        """
        self._path = Path(path)

    def export(self, spans: Sequence[ReadableSpan]) -> SpanExportResult:
        """Append a batch of spans to the JSONL file.

        Args:
            spans: Completed spans to serialize and write.

        Returns:
            `SUCCESS` once all spans have been written.
        """
        with self._path.open("a") as f:
            for span in spans:
                f.write(span.to_json(indent=None) + "\n")
        return SpanExportResult.SUCCESS

    def shutdown(self) -> None:
        """Shut down the exporter.

        No resources are held open between exports, so this is a no-op.
        """


# The TracerProvider is the entry point of the tracing API — it creates tracers
# and owns the pipeline that processes the spans they produce.
provider = TracerProvider()

# A span processor decides how finished spans reach the exporter. SimpleSpanProcessor
# exports each span synchronously as soon as it ends — straightforward and ordered,
# ideal for notebooks/debugging (use BatchSpanProcessor in production for throughput).
provider.add_span_processor(SimpleSpanProcessor(JSONLFileExporter("traces.jsonl")))

# Register this provider as the global default so all instrumentation uses it.
trace.set_tracer_provider(provider)

# Auto-instrument LangChain — do this once, before building/invoking the agent
LangChainInstrumentor().instrument()

# Define agent

In [ ]:
@tool
def think(reflection: str) -> str:
    """Pause to reflect on progress before deciding the next action.

    Use at natural checkpoints — after a search, a function, a subtask, or any
    step whose outcome should shape what you do next.

    When to use:
    - After a discrete step: What did I just learn or produce?
    - Before deciding next steps: Do I have what I need to move forward, or to finish?
    - When assessing gaps: What is still unknown, undecided, or incomplete?
    - When complexity rises: Am I on the right path, or should I change approach?
    - Before concluding: Can I deliver a complete, correct result now?

    When NOT to use:
    - The next action is already obvious.
    - The step was mechanical (formatting, a rename, a trivial edit).
    - You just reflected and nothing material has changed.

    Reflection should consider any of the following that apply:
    1. Current state - What concrete progress, findings, or output do I have?
    2. Gaps - What is still unknown, undecided, or incomplete?
    3. Quality - Is the work so far sound and sufficient?
    4. Next move - Continue, pivot, or finish?

    Keep it to 1-4 sentences. Go longer only when the decision is genuinely hard.

    Args:
        reflection (str): Your reflection on progress, gaps, and next steps

    Returns:
        Confirmation that reflection was recorded for decision-making
    """
    logger.info("🧠 thinking...")
    return f"Reflection recorded: {reflection}"


type Numeric = int | float
type Operation = Literal["add", "subtract", "multiply", "divide"]


@tool
def calculator(  # noqa: C901
    operation: Operation,
    a: Numeric,
    b: Numeric,
) -> Numeric | str:
    """Apply an arithmetic operation to two numbers and return result.

    Args:
        operation (Operation): Operation to apply.
        a (Numeric): First number
        b (Numeric): Second number

    Returns:
        Numeric | str: Result of arithmetic operation applied to the two input numbers, or error message.
    """
    logger.info("🔢 calculator: {operation}({a}, {b})", operation=operation, a=a, b=b)
    match operation:
        case "add":
            return a + b
        case "subtract":
            return a - b
        case "multiply":
            return a * b
        case "divide":
            if b == 0:
                return "error: Cannot divide by 0."
            return a / b
        case _:
            return f"error: {operation} not supported."

In [ ]:
agent = create_agent(
    model=get_chat_model(model_name=ModelName.CLAUDE_HAIKU),
    tools=[calculator, think],
    system_prompt="""
    You are a helpful question answering assistant. Use your tools
    to answer the user's question as accurately as possible.

    Use the calculator for math, don't guess.
    """,
    checkpointer=InMemorySaver(),
)

# Multi-turn conversation

Run a conversation — traces save automatically

In [ ]:
# Define conversation thread id
config = {"configurable": {"thread_id": "thread-1"}}

In [ ]:
# Turn 1
response = agent.invoke({"messages": {"role": "user", "content": "hi!"}}, config=config)
print(get_last_message(response).content)

# Turn 2
response = agent.invoke({"messages": {"role": "user", "content": "are you a thinking machine?"}}, config=config)
print(get_last_message(response).content)


# Turn 3
response = agent.invoke({"messages": {"role": "user", "content": "ok, what's 2+2?"}}, config=config)
print(get_last_message(response).content)

# Turn 4
response = agent.invoke(
    {"messages": {"role": "user", "content": "that was easy, what about 12,345 * 532,323 - 2^13"}}, config=config
)
print(get_last_message(response).content)
# Expected correct answer 6,571,519,243

# Turn 5
question = """
Three friends — Alex, Ben, and Chloe — perform the following six transactions, in exact order:

Alex gives Ben an amount equal to 20% of Alex's current money
Ben gives Chloe an amount equal to 25% of Ben's current money
Chloe gives Alex an amount equal to 40% of Chloe's current money
Alex gives Ben an amount equal to 50% of Alex's current money
Ben gives Chloe an amount equal to 25% of Ben's current money
Chloe gives Alex an amount equal to 50% of Chloe's current money
After all six transactions:

Alex has $99
Ben has $99
Chloe has $42
Each person started with a whole-dollar amount. How much did each start with?
"""

response = agent.invoke(
    {"messages": {"role": "user", "content": f"ok, you're pretty good...now try this: {question}"}}, config=config
)
print(get_last_message(response).content)

# Expected correct answer
# Alex: $100, Ben: $80, Chloe: $60 (total $240)

# Load saved traces for analysis

Each line of `traces.jsonl` is one OpenTelemetry span serialized to JSON. We read the file into a Polars DataFrame and reshape it into a flat, analysis-friendly table — one row per span.

The pipeline does four things:

1. **Read** the newline-delimited JSON. `infer_schema_length=None` scans *every* row before settling on a schema, so columns that only appear on some spans (e.g. token counts on LLM spans) aren't missed.
2. **Unnest `context`** — the span's `context` object holds `trace_id`, `span_id`, and `trace_state`; `unnest` lifts those into top-level columns so spans can be grouped by trace.
3. **Parse timestamps & compute duration** — `start_time`/`end_time` are ISO-8601 strings; we parse them to datetimes and subtract to get each span's wall-clock `duration_ms`.
4. **Pull span attributes** — the `attributes` struct uses flat, dotted [OpenInference](https://github.com/Arize-ai/openinference) keys. We surface the ones we care about (span kind, model, token counts) as their own columns.

> **Note on tokens:** OpenInference emits `llm.token_count.prompt` and `llm.token_count.completion` but *not* a total, so we derive `total_tokens` ourselves. These fields exist only on `LLM` spans; on other span kinds (e.g. `CHAIN`) they're `null`.

In [ ]:
# Load traces as dataframe
df = (
    # Read newline-delimited spans. infer_schema_length=None scans all rows so
    # fields that appear on only some spans (e.g. token counts) aren't dropped.
    pl
    .read_ndjson("traces.jsonl", infer_schema_length=None)
    # Lift the span context object into top-level columns for grouping by trace.
    .unnest("context")  # -> trace_id, span_id, trace_state
    # Parse ISO-8601 timestamp strings into datetimes.
    .with_columns(
        start=pl.col("start_time").str.to_datetime("%Y-%m-%dT%H:%M:%S%.fZ"),
        end=pl.col("end_time").str.to_datetime("%Y-%m-%dT%H:%M:%S%.fZ"),
    )
    # Wall-clock duration of each span, in milliseconds.
    .with_columns(duration_ms=(pl.col("end") - pl.col("start")).dt.total_microseconds() / 1000)
    # Surface the OpenInference attributes we care about as their own columns.
    # The attributes struct uses flat, dotted keys; token/model fields are
    # populated only on LLM spans and are null elsewhere.
    .with_columns(
        thread_id=pl.col("attributes").struct.field("session.id"),
        span_kind=pl.col("attributes").struct.field("openinference.span.kind"),
        model=pl.col("attributes").struct.field("llm.model_name"),
        prompt_tokens=pl.col("attributes").struct.field("llm.token_count.prompt"),
        completion_tokens=pl.col("attributes").struct.field("llm.token_count.completion"),
    )
    # OpenInference emits prompt/completion counts but not a total — derive it.
    .with_columns(total_tokens=pl.col("prompt_tokens") + pl.col("completion_tokens"))
)

# Order traces chronologically and assign a 1-based turn number.
turn_order = (
    df.group_by("trace_id").agg(turn_start=pl.col("start").min()).sort("turn_start").with_row_index("turn", offset=1)
)

df = df.join(turn_order, on="trace_id")

In [ ]:
# Unique turns (traces)
df.get_column("trace_id").unique()

### Total duration per turn (trace)

In [ ]:
trace_duration = (
    df
    .group_by("turn")
    .agg(total_sec=(pl.col("end").max() - pl.col("start").min()).dt.total_microseconds() / 1e6)
    .sort("turn")
)

trace_duration

### Tool calls per turn

In [ ]:
tool_calls_per_turn = (
    df
    .filter(pl.col("span_kind") == "TOOL")
    .group_by("trace_id")
    .agg(
        tool_calls=pl.len(),
        by_tool=pl.col("name").value_counts(),
    )
    .join(turn_order, on="trace_id")
    .sort("turn")
    .select("turn", "tool_calls", "by_tool")
)
tool_calls_per_turn

### Filter to subtraction calculations
Filter to traces with a tool call: `calculator(operation="subtract")`

In [ ]:
subtract_traces = (
    df
    .filter(
        # Filter to tool call spans
        (pl.col("span_kind") == "TOOL")
        # Filter to "calculator" tool
        & (pl.col("name") == "calculator")
        # Filter to calculator tool calls with arg: operation == subtract
        & (pl.col("attributes").struct.field("input.value").str.json_path_match("$.operation") == "subtract")
    )
    .get_column("trace_id")
    .unique()
)

df_subtract = df.filter(pl.col("trace_id").is_in(subtract_traces))

### Input/output of think tool calls

In [ ]:
think_calls = (
    df
    .filter((pl.col("span_kind") == "TOOL") & (pl.col("name") == "think"))
    .with_columns(
        reflection=pl.col("attributes").struct.field("input.value"),
        output=pl.col("attributes").struct.field("output.value").str.json_path_match("$.data.content"),
    )
    .select("turn", "trace_id", "reflection", "output")
    .sort("turn")
)
think_calls

### Token cost per think call

In [ ]:
# All LLM spans, ordered within each trace, tagged with the next span's name
llm = (
    df
    .filter(pl.col("span_kind") == "LLM")
    .sort("trace_id", "start")
    .select("trace_id", "start", "completion_tokens", "prompt_tokens")
)

# The think tool spans
think_spans = df.filter((pl.col("span_kind") == "TOOL") & (pl.col("name") == "think")).select(
    "turn", "trace_id", think_start="start"
)

# For each think call, grab the LLM span that immediately precedes it (the one
# that generated the tool call) via an as-of join on start time within the trace.
think_cost = (
    think_spans
    .sort("think_start")
    .join_asof(
        llm.sort("start"),
        left_on="think_start",
        right_on="start",
        by="trace_id",
        strategy="backward",  # nearest LLM span at or before the think call
    )
    .select(
        "turn",
        "trace_id",
        gen_completion_tokens=pl.col("completion_tokens"),  # cost to produce the call
        gen_prompt_tokens=pl.col("prompt_tokens"),
    )
)
think_cost